# 🦷 Cavity Detection & Cropping — YOLOv12

This notebook uses the pre-trained **YOLOv12** model from the
[YOLOv12-Tooth-Cavity-Detection](../../YOLOv12-Tooth-Cavity-Detection/) project
to:

1. Run inference on the dental X-ray dataset.
2. Identify teeth predicted as **cavity** (carious).
3. Save each detected carious tooth as a separate **cropped image** in an output
   folder for downstream analysis (e.g., classification, review).

## 1  Imports & Configuration

In [1]:
import os
import sys
import glob
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.notebook import tqdm

from ultralytics import YOLO

print(f"Python  : {sys.version}")
print(f"OpenCV  : {cv2.__version__}")

Python  : 3.11.16 (main, Aug 27 2026, 14:44:21) [GCC 14.3.0]
OpenCV  : 5.0.0


In [2]:
# ── Paths ──────────────────────────────────────────────────────────────
# YOLOv12 model weights (trained for tooth-cavity detection)
MODEL_PATH = Path("../../YOLOv12-Tooth-Cavity-Detection/models/best.pt")

# Root directory containing the dental images
# (sub-folders: Upper_Occlusal, Lower_Left, etc.)
DATA_ROOT = Path("../data/Dental_Image_Dataset_mixed")

# Output directory for the cropped cavity images
CROPS_DIR = Path("output/cavity_crops")

# ── Detection settings ─────────────────────────────────────────────────
CONFIDENCE_THRESHOLD = 0.25   # minimum confidence to keep a detection
CAVITY_CLASS_ID      = 0      # class index for 'cavity' in the model
CLASS_NAMES          = ["cavity", "normal"]
CROP_PADDING         = 5      # extra pixels around the bounding box
IMAGE_EXTENSIONS     = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".tif"}

# ── Create output directory ────────────────────────────────────────────
CROPS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Model weights : {MODEL_PATH}  (exists={MODEL_PATH.exists()})")
print(f"Data root     : {DATA_ROOT}   (exists={DATA_ROOT.exists()})")
print(f"Crops output  : {CROPS_DIR}")

Model weights : ../../YOLOv12-Tooth-Cavity-Detection/models/best.pt  (exists=True)
Data root     : ../data/Dental_Image_Dataset_mixed   (exists=True)
Crops output  : output/cavity_crops


## 2  Load the YOLOv12 Model

In [3]:
model = YOLO(str(MODEL_PATH))
print(f"✅ Model loaded — class names: {model.names}")

✅ Model loaded — class names: {0: 'cavity', 1: 'normal'}


## 3  Discover All Images in the Dataset

In [4]:
image_paths = sorted(
    p for p in DATA_ROOT.rglob("*")
    if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
)
print(f"Found {len(image_paths):,} images across sub-folders.")

# Quick breakdown per sub-folder
from collections import Counter
folder_counts = Counter(p.parent.name for p in image_paths)
for folder, count in sorted(folder_counts.items()):
    print(f"  {folder:20s} → {count:>5,} images")

Found 6,378 images across sub-folders.
  Lower_Front          →   191 images
  Lower_Left           → 1,065 images
  Lower_Occlusal       → 1,224 images
  Lower_Right          →   581 images
  Upper_Front          →   623 images
  Upper_Left           →   539 images
  Upper_Occlusal       → 1,543 images
  Upper_Right          →   612 images


## 4  Run Inference & Crop Carious Teeth

For each image we:
1. Run the YOLO model.
2. Filter detections to **cavity** class only.
3. Crop the bounding-box region (with a small padding) from the *original* image.
4. Save each crop as `<subfolder>_<image_stem>_crop<N>.jpg` in the output folder.

A summary CSV (`crops_index.csv`) is also written to make it easy to trace each
crop back to its source image.

In [5]:
import csv

csv_path = CROPS_DIR / "crops_index.csv"
csv_fields = [
    "crop_filename", "source_image", "subfolder",
    "confidence", "x1", "y1", "x2", "y2",
]

total_cavities = 0
images_with_cavities = 0

with open(csv_path, "w", newline="") as csv_file:
    writer = csv.DictWriter(csv_file, fieldnames=csv_fields)
    writer.writeheader()

    for img_path in tqdm(image_paths, desc="Processing images"):
        # Read the original image (BGR)
        img_bgr = cv2.imread(str(img_path))
        if img_bgr is None:
            print(f"⚠️  Could not read: {img_path}")
            continue

        h, w = img_bgr.shape[:2]

        # Run YOLO inference
        results = model.predict(
            source=str(img_path),
            conf=CONFIDENCE_THRESHOLD,
            verbose=False,
        )

        boxes = results[0].boxes
        if boxes is None or len(boxes) == 0:
            continue

        subfolder = img_path.parent.name
        stem = img_path.stem
        crop_idx = 0

        for box in boxes:
            cls_id = int(box.cls)
            if cls_id != CAVITY_CLASS_ID:
                continue  # skip non-cavity detections

            confidence = float(box.conf)
            x1, y1, x2, y2 = map(int, box.xyxy[0])

            # Apply padding (clamped to image bounds)
            x1 = max(0, x1 - CROP_PADDING)
            y1 = max(0, y1 - CROP_PADDING)
            x2 = min(w, x2 + CROP_PADDING)
            y2 = min(h, y2 + CROP_PADDING)

            # Crop from the original (un-annotated) image
            crop = img_bgr[y1:y2, x1:x2]
            if crop.size == 0:
                continue

            crop_idx += 1
            crop_filename = f"{subfolder}_{stem}_crop{crop_idx}.jpg"
            crop_save_path = CROPS_DIR / crop_filename
            cv2.imwrite(str(crop_save_path), crop)

            writer.writerow({
                "crop_filename": crop_filename,
                "source_image": str(img_path),
                "subfolder": subfolder,
                "confidence": f"{confidence:.4f}",
                "x1": x1, "y1": y1, "x2": x2, "y2": y2,
            })

        if crop_idx > 0:
            total_cavities += crop_idx
            images_with_cavities += 1

print(f"\n✅ Done!")
print(f"   Images processed      : {len(image_paths):,}")
print(f"   Images with cavities  : {images_with_cavities:,}")
print(f"   Total cavity crops    : {total_cavities:,}")
print(f"   Crops saved to        : {CROPS_DIR.resolve()}")
print(f"   Index CSV             : {csv_path.resolve()}")

Processing images:   0%|          | 0/6378 [00:00<?, ?it/s]


✅ Done!
   Images processed      : 6,378
   Images with cavities  : 6,204
   Total cavity crops    : 40,334
   Crops saved to        : /home/harlet/Projects/Technical_Seminar/version_3/output/cavity_crops
   Index CSV             : /home/harlet/Projects/Technical_Seminar/version_3/output/cavity_crops/crops_index.csv


## 5  Preview Sample Crops

In [ ]:
crop_files = sorted(CROPS_DIR.glob("*.jpg"))
n_preview = min(16, len(crop_files))

if n_preview == 0:
    print("No cavity crops were generated.")
else:
    cols = 4
    rows = (n_preview + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    axes = np.asarray(axes).ravel()

    for i, ax in enumerate(axes):
        if i < n_preview:
            img = Image.open(crop_files[i])
            ax.imshow(img)
            ax.set_title(crop_files[i].name, fontsize=8)
        ax.axis("off")

    fig.suptitle("Sample Cavity Crops", fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()

## 6  Preview Annotated Detections on a Sample Image

Pick a random image that has cavity detections and display the annotated result
side-by-side with its crops.

In [ ]:
import pandas as pd
import random

df = pd.read_csv(csv_path)
if len(df) == 0:
    print("No detections to visualise.")
else:
    # Pick a random source image that had cavities
    sample_src = random.choice(df["source_image"].unique())
    sample_rows = df[df["source_image"] == sample_src]

    img_bgr = cv2.imread(sample_src)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # Draw bounding boxes
    annotated = img_rgb.copy()
    for _, row in sample_rows.iterrows():
        x1, y1, x2, y2 = int(row.x1), int(row.y1), int(row.x2), int(row.y2)
        cv2.rectangle(annotated, (x1, y1), (x2, y2), (255, 0, 0), 3)
        label = f"cavity {row.confidence:.2f}"
        cv2.putText(annotated, label, (x1, y1 - 8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)

    n_crops = len(sample_rows)
    fig, axes = plt.subplots(1, 1 + n_crops, figsize=(6 * (1 + n_crops), 6))
    if not isinstance(axes, np.ndarray):
        axes = [axes]

    axes[0].imshow(annotated)
    axes[0].set_title(f"Source: {Path(sample_src).name}", fontsize=10)
    axes[0].axis("off")

    for i, (_, row) in enumerate(sample_rows.iterrows(), start=1):
        crop_img = Image.open(CROPS_DIR / row.crop_filename)
        axes[i].imshow(crop_img)
        axes[i].set_title(f"Crop {i}  (conf={row.confidence:.2f})", fontsize=10)
        axes[i].axis("off")

    plt.tight_layout()
    plt.show()

## 7  Summary Statistics

In [ ]:
if len(df) > 0:
    print(f"Total cavity crops saved : {len(df):,}")
    print(f"Unique source images     : {df['source_image'].nunique():,}")
    print(f"\nConfidence statistics:")
    print(df["confidence"].describe().to_string())

    # Breakdown per sub-folder
    print("\nCavity crops per sub-folder:")
    print(df.groupby("subfolder").size().sort_values(ascending=False).to_string())

    # Confidence distribution
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.hist(df["confidence"], bins=30, edgecolor="white", color="#e74c3c")
    ax.set_xlabel("Confidence")
    ax.set_ylabel("Count")
    ax.set_title("Cavity Detection Confidence Distribution")
    plt.tight_layout()
    plt.show()
else:
    print("No cavity detections found in the dataset.")